# <font color='blue'> Chapter 54: Proximal Policy Optimization (PPO) </font>

Policy Gradient and Actor–Critic methods optimize policies directly by following the gradient of the expected return.

However,

large policy updates can cause learning to become unstable.

If the policy changes too much after a single update,

the agent may suddenly begin taking very different actions,

leading to a dramatic decrease in performance.

Proximal Policy Optimization (PPO) solves this problem by **restricting how much the policy may change during each optimization step**.

Rather than allowing arbitrarily large updates,

PPO performs many small, stable improvements,

making it one of the most successful Reinforcement Learning algorithms developed for deep neural networks.

---



# <font color='orange'> 1. Motivation </font>

Suppose a robot has learned to walk reasonably well.

```
Current Policy

↓

Walk Forward

↓

Small Improvement

↓

Walk Better
```

Now imagine making an extremely large update.

```
Current Policy

↓

Huge Update

↓

Robot Falls
```

Large policy changes often destroy previously learned behaviour.

PPO prevents this by ensuring that policy updates remain **proximal** (close) to the previous policy.

---



# <font color='orange'> 2. Why Ordinary Policy Gradients Can Fail </font>

Policy Gradient methods update the parameters using

$$
\theta
\leftarrow
\theta
+
\alpha
\nabla_\theta J(\theta).
$$

Nothing in this update prevents

the policy from changing dramatically.

Large updates may

- destabilize training,
- increase variance,
- reduce performance,
- cause catastrophic forgetting.

PPO introduces a mechanism that discourages excessively large policy changes.

---



# <font color='orange'> 3. Old Policy vs New Policy </font>

Suppose

the old policy is

$$
\pi_{\theta_{\mathrm{old}}}.
$$

After optimization,

the new policy becomes

$$
\pi_\theta.
$$

Rather than comparing parameters directly,

PPO compares the probabilities assigned to the same action under the two policies.

This comparison is captured by the **probability ratio**.

---



# <font color='orange'> 4. Probability Ratio </font>

The probability ratio is defined as

$$
\boxed{
r_t(\theta)
=
\frac{\pi_\theta(a_t|s_t)}
{\pi_{\theta_{\mathrm{old}}}(a_t|s_t)}.
}
$$

Interpretation

- \(r_t = 1\): the policy has not changed.
- \(r_t > 1\): the action has become more likely.
- \(r_t < 1\): the action has become less likely.

This ratio measures how much the new policy differs from the old one.

---



# <font color='orange'> 5. Why Clipping is Needed </font>

Suppose

the policy suddenly changes,

making

$$
r_t=3.
$$

This would greatly increase the importance of a single update,

possibly destabilizing training.

Instead,

PPO clips the ratio,

preventing excessively large policy changes.

---



# <font color='orange'> 6. Clipping Function </font>

Suppose

$$
\varepsilon=0.2.
$$

Then

```
Allowed Range

0.8

↓

1.2
```

If

$$
r_t
$$

moves outside this interval,

it is clipped back into the permitted range.

This keeps policy updates small and stable.

---



# <font color='orange'> 7. The PPO Objective </font>

The clipped surrogate objective is

$$
\boxed{
L^{\mathrm{CLIP}}(\theta)
=
\mathbb E
\left[
\min
\left(
r_t(\theta)A_t,
\;
\operatorname{clip}
(r_t(\theta),1-\varepsilon,1+\varepsilon)
A_t
\right)
\right].
}
$$

where

- \(A_t\) is the advantage estimate,
- \(r_t\) is the probability ratio.

The minimum operator ensures that updates never become overly aggressive.

---



# <font color='orange'> 8. Why Does Clipping Work? </font>

Suppose

the advantage is positive.

Increasing the action probability is beneficial.

However,

after a certain point,

additional increases are ignored by the clipped objective.

Similarly,

if the advantage is negative,

the policy is discouraged from decreasing the probability too aggressively.

Clipping therefore creates a region where optimization is allowed,

while preventing excessively large policy updates.

---



# <font color='orange'> 9. PPO Training Pipeline </font>

A typical PPO iteration proceeds as follows.

```
Collect Trajectories

↓

Compute Returns

↓

Estimate Advantages

↓

Compute Probability Ratios

↓

Compute PPO Loss

↓

Mini-batch Gradient Ascent

↓

Update Policy

↓

Repeat
```

Unlike REINFORCE,

the same collected experience is often reused for several optimization epochs.

This improves sample efficiency.

---



# <font color='orange'> 10. Mini-Batch Optimization </font>

Instead of updating

after every episode,

PPO typically

- collects many trajectories,
- divides them into mini-batches,
- performs multiple gradient updates.

This resembles standard deep learning optimization and makes efficient use of collected experience.

---



# <font color='orange'> 11. PPO and the Critic </font>

PPO is usually implemented as an

**Actor–Critic algorithm**.

The Actor optimizes the clipped policy objective.

The Critic estimates

$$
V(s),
$$

which is used to compute the advantage

$$
A_t.
$$

Thus,

the Critic continues to reduce variance,

while PPO stabilizes policy optimization.

---



# <font color='orange'> 12. Advantages </font>

PPO

- is simple to implement,
- is computationally efficient,
- provides stable learning,
- performs well across many domains,
- works with high-dimensional observations,
- supports continuous and discrete action spaces.

These advantages explain its widespread adoption.

---



# <font color='orange'> 13. Applications </font>

PPO has been successfully applied to

- robotics,
- autonomous driving,
- game playing,
- industrial process control,
- recommendation systems,
- reinforcement learning from human feedback (RLHF).

For several years, PPO was one of the primary algorithms used to optimize language models during the RLHF stage.

---



# <font color='orange'> 14. Limitations </font>

Although highly successful,

PPO is not perfect.

It

- requires careful hyperparameter tuning,
- may require large amounts of interaction data,
- is not guaranteed to find the global optimum,
- can still become unstable if advantage estimates are poor.

More recent methods,

such as Direct Preference Optimization (DPO),

simplify certain alignment tasks by avoiding reinforcement learning altogether.

---

# <font color='red'> 15. Mathematical Foundations </font>

Probability ratio

$$
\boxed{
r_t(\theta)
=
\frac{\pi_\theta(a_t|s_t)}
{\pi_{\theta_{\mathrm{old}}}(a_t|s_t)}.
}
$$

Advantage

$$
\boxed{
A_t
=
Q(s_t,a_t)-V(s_t).
}
$$

Clipped objective

$$
\boxed{
L^{\mathrm{CLIP}}(\theta)
=
\mathbb E
\left[
\min
\left(
r_tA_t,
\;
\operatorname{clip}
(r_t,1-\varepsilon,1+\varepsilon)
A_t
\right)
\right].
}
$$

Policy parameters are updated using gradient ascent,

$$
\boxed{
\theta
\leftarrow
\theta
+
\alpha
\nabla_\theta
L^{\mathrm{CLIP}}(\theta).
}
$$

The Critic is typically trained by minimizing a value-function loss,

often the mean squared error between predicted values and estimated returns.

---



# <font color='orange'> 16. PPO vs REINFORCE vs Actor–Critic </font>

| Algorithm | Main Idea | Stability |
|:---|:---|:---|
| REINFORCE | Monte Carlo policy gradients | Low |
| Actor–Critic | Policy + value estimation | Moderate |
| PPO | Actor–Critic with clipped updates | High |

PPO combines variance reduction from the Critic with constrained policy updates, leading to robust performance across many tasks.

---



# <font color='orange'> 17. Common Misconceptions </font>

### Misconception 1

> PPO always finds the globally optimal policy.

**False.**

PPO performs local optimization using gradient ascent. Like most deep learning methods, it is not guaranteed to reach the global optimum.

---

### Misconception 2

> Clipping prevents all policy changes.

**False.**

Clipping limits excessively large updates but still allows the policy to improve through many small optimization steps.

---

### Misconception 3

> PPO no longer needs a Critic.

**False.**

Most practical implementations of PPO use an Actor–Critic architecture, where the Critic estimates the value function used to compute advantages.

---

# <font color='purple'> 18. Conceptual Summary </font>

| Concept | Description |
|:---|:---|
| PPO | Stable policy optimization algorithm based on Actor–Critic methods |
| Probability Ratio | Measures the change between the new and old policy |
| Clipping | Limits excessively large policy updates |
| Clipped Surrogate Objective | Optimization objective that stabilizes learning |
| Advantage Function | Measures whether an action was better or worse than expected |
| Mini-Batch Optimization | Reuses collected trajectories efficiently |
| Actor–Critic | Architecture commonly used to implement PPO |

> **Key Insight:** Proximal Policy Optimization improves policy gradient methods by preventing excessively large policy updates. Instead of allowing unrestricted optimization, PPO compares the new policy with the previous policy using a probability ratio and clips updates that move too far from the original behaviour. Combined with an Actor–Critic architecture and advantage estimation, PPO achieves stable, sample-efficient learning and has become one of the most widely used reinforcement learning algorithms in robotics, game playing, and language model alignment.